# A7-L2 practical: Write run log records for a successful and a failed run

Platform: Databricks Free Edition.

Run this in Databricks Free Edition. It creates a run log table with an explicit schema, runs one step that succeeds and one that fails, and records both. It also shows the inference error the reference code hits.


In [ ]:
import uuid
from collections.abc import Callable
from datetime import datetime, timezone

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import types as T

CATALOG = spark.catalog.currentCatalog()
SCHEMA = f"{CATALOG}.learning_a7"
RUN_LOG = f"{SCHEMA}.pipeline_run_log"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

RUN_LOG_SCHEMA = T.StructType([
    T.StructField("run_id", T.StringType(), False),
    T.StructField("pipeline_name", T.StringType(), False),
    T.StructField("rows_processed", T.LongType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("started_at", T.TimestampType(), False),
    T.StructField("finished_at", T.TimestampType(), False),
])

In [ ]:
def run_logged(spark: SparkSession, name: str, step: Callable[[], int]) -> None:
    """Run a step, record the outcome in the run log, and re-raise failures."""
    run_id, started = str(uuid.uuid4()), datetime.now(timezone.utc)
    status, rows, error = "FAILED", 0, None
    try:
        rows = step()
        status = "SUCCESS"
    except Exception as err:
        error = f"{type(err).__name__}: {err}"
        raise
    finally:
        row = (run_id, name, rows, status, error, started, datetime.now(timezone.utc))
        spark.createDataFrame([row], RUN_LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(RUN_LOG)

In [ ]:
run_logged(spark, "nat_silver_pipeline", lambda: 1832)
try:
    run_logged(spark, "nat_gold_pipeline", lambda: 1 // 0)
except ZeroDivisionError:
    print("the failure was re-raised, and recorded")
display(spark.table(RUN_LOG).orderBy("started_at"))

Now the reference approach: a dict with `error_message=None` and no schema.

In [ ]:
try:
    spark.createDataFrame([{"run_id": "x", "error_message": None}])
except Exception as err:
    print(type(err).__name__, str(err)[:120])